In [ ]:
# Per-class analysis of HEST evaluation results
# Compare trimodal vs bimodal model performance at the class level for 10 organs/9 cancers
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import numpy as np
from collections import defaultdict
import re
import glob

print("Starting per-class HEST analysis...")

In [ ]:
snakemake.input

In [ ]:
# Load HEST evaluation results and individual CLIP scores
results = []
clip_scores_all = []

# Parse input files to extract model type and dataset
for result_file in snakemake.input:
    try:
        df = pd.read_csv(result_file)

        # Extract model type and dataset from file path
        path_parts = Path(result_file).parts

        # Find the log directory name that contains model and dataset info
        log_dir = None
        for part in path_parts:
            if part.startswith("hest_eval___"):
                log_dir = part
                break

        if log_dir:
            # Parse: hest_eval___spotwhisperer_{model}___{dataset}
            parts = log_dir.split("___")
            if len(parts) >= 3:
                model_name = parts[1].replace("spotwhisperer_", "")
                dataset = parts[2]

                # Determine model type based on dataset combination
                if model_name == "cellxgene_census__archs4_geo__hest1k__quilt1m":
                    model_type = "trimodal"
                elif model_name == "cellxgene_census__archs4_geo":
                    model_type = "bimodal_mismatch1"
                elif model_name == "hest1k":
                    model_type = "bimodal_matching"  # HEST matches with hest1k
                elif model_name == "quilt1m":
                    model_type = "bimodal_mismatch2"
                else:
                    model_type = "unknown"

                # Add metadata to metrics
                df["model_type"] = model_type
                df["model_name"] = model_name
                df["dataset"] = dataset
                df["result_file"] = str(result_file)

                results.append(df)

                # Look for individual CLIP scores file in the same directory
                clip_scores_file = (
                    Path(result_file).parent / "test_individual_clip_scores.csv"
                )
                if clip_scores_file.exists():
                    try:
                        clip_df = pd.read_csv(clip_scores_file)
                        clip_df["model_type"] = model_type
                        clip_df["model_name"] = model_name
                        clip_df["dataset"] = dataset
                        clip_scores_all.append(clip_df)
                    except Exception as e:
                        print(f"Error loading CLIP scores from {clip_scores_file}: {e}")

    except Exception as e:
        print(f"Error loading {result_file}: {e}")

if results:
    combined_df = pd.concat(results, ignore_index=True)
    print(f"Loaded {len(combined_df)} metric entries from {len(results)} files")
    print(combined_df.head())
else:
    print("No results loaded!")
    combined_df = pd.DataFrame()

if clip_scores_all:
    clip_scores_df = pd.concat(clip_scores_all, ignore_index=True)
    print(
        f"Loaded {len(clip_scores_df)} CLIP score entries from {len(clip_scores_all)} files"
    )
else:
    print("No CLIP scores loaded!")
    clip_scores_df = pd.DataFrame()

In [ ]:
# Create comparison between trimodal and bimodal models
# Focus on retrieval metrics: test_retrieval/transcriptome_image/rocauc_macroAvg

model_comparison = pd.DataFrame()

if not combined_df.empty:
    # Filter for the main retrieval metric
    main_metric = "test_retrieval/transcriptome_image/rocauc_macroAvg"

    if main_metric in combined_df.columns:
        metric_data = combined_df[["dataset", "model_type", main_metric]].copy()
        metric_data = metric_data.dropna()

        # Pivot to get model types as columns
        pivot_df = metric_data.pivot_table(
            index="dataset", columns="model_type", values=main_metric, aggfunc="mean"
        ).reset_index()

        # Calculate improvement (trimodal - bimodal_matching)
        if "trimodal" in pivot_df.columns and "bimodal_matching" in pivot_df.columns:
            pivot_df["improvement"] = (
                pivot_df["trimodal"] - pivot_df["bimodal_matching"]
            )
            pivot_df["relative_improvement"] = (
                pivot_df["improvement"] / pivot_df["bimodal_matching"] * 100
            )

            # Add class labels (organ/cancer types)
            pivot_df["class_type"] = pivot_df["dataset"].apply(
                lambda x: (
                    "Cancer"
                    if x
                    in [
                        "IDC",
                        "PRAD",
                        "PAAD",
                        "SKCM",
                        "COAD",
                        "READ",
                        "CCRCC",
                        "HCC",
                        "LUNG",
                    ]
                    else "Organ"
                )
            )

            model_comparison = pivot_df
            print("Model comparison created:")
            print(model_comparison.head(10))
        else:
            print("Required model types not found in data")
    else:
        print(f"Main metric {main_metric} not found in data")
        print("Available columns:", combined_df.columns.tolist())

In [ ]:
# Create CLIP scores comparison if available
clip_comparison = pd.DataFrame()

if not clip_scores_df.empty:
    # Aggregate CLIP scores by dataset and model type
    clip_agg = (
        clip_scores_df.groupby(["dataset", "model_type"])[
            ["clip_score_left_right", "clip_score_right_left"]
        ]
        .mean()
        .reset_index()
    )

    # Use the average of both directions as overall CLIP score
    clip_agg["avg_clip_score"] = (
        clip_agg["clip_score_left_right"] + clip_agg["clip_score_right_left"]
    ) / 2

    # Pivot to compare model types
    clip_pivot = clip_agg.pivot_table(
        index="dataset", columns="model_type", values="avg_clip_score", aggfunc="mean"
    ).reset_index()

    if "trimodal" in clip_pivot.columns and "bimodal_matching" in clip_pivot.columns:
        clip_pivot["clip_improvement"] = (
            clip_pivot["trimodal"] - clip_pivot["bimodal_matching"]
        )
        clip_pivot["clip_relative_improvement"] = (
            clip_pivot["clip_improvement"] / clip_pivot["bimodal_matching"] * 100
        )
        clip_comparison = clip_pivot
        print("CLIP scores comparison created:")
        print(clip_comparison.head())

In [ ]:
if not model_comparison.empty:
    # Create visualization
    fig, axes = plt.subplots(2, 2, figsize=(20, 16))
    fig.suptitle(
        "HEST Per-Class Performance: Trimodal vs Bimodal Models\n(10 Organs/9 Cancers)",
        fontsize=16,
    )

    # Plot 1: Top 10 improving datasets
    top_improving = model_comparison.nlargest(10, "improvement")
    if not top_improving.empty:
        y_pos = np.arange(len(top_improving))
        colors = ["green" if x >= 0 else "red" for x in top_improving["improvement"]]
        axes[0, 0].barh(y_pos, top_improving["improvement"], color=colors, alpha=0.7)
        axes[0, 0].set_yticks(y_pos)
        axes[0, 0].set_yticklabels(
            [
                f"{row['dataset']} ({row['class_type']})"
                for _, row in top_improving.iterrows()
            ]
        )
        axes[0, 0].set_xlabel("ROC-AUC Improvement")
        axes[0, 0].set_title(
            "Top 10 Datasets: ROC-AUC Improvement\n(Trimodal - Bimodal)"
        )
        axes[0, 0].axvline(x=0, color="black", linestyle="--", alpha=0.5)
        axes[0, 0].grid(True, alpha=0.3)

    # Plot 2: Performance comparison by class type (Cancer vs Organ)
    class_comparison = model_comparison.groupby("class_type")[
        ["trimodal", "bimodal_matching", "improvement"]
    ].mean()
    if not class_comparison.empty:
        x = np.arange(len(class_comparison))
        width = 0.35
        axes[0, 1].bar(
            x - width / 2,
            class_comparison["bimodal_matching"],
            width,
            label="Bimodal",
            alpha=0.8,
        )
        axes[0, 1].bar(
            x + width / 2,
            class_comparison["trimodal"],
            width,
            label="Trimodal",
            alpha=0.8,
        )
        axes[0, 1].set_xlabel("Class Type")
        axes[0, 1].set_ylabel("Mean ROC-AUC")
        axes[0, 1].set_title("Mean ROC-AUC by Class Type")
        axes[0, 1].set_xticks(x)
        axes[0, 1].set_xticklabels(class_comparison.index)
        axes[0, 1].legend()
        axes[0, 1].grid(True, alpha=0.3)

    # Plot 3: All datasets comparison
    sorted_data = model_comparison.sort_values("improvement", ascending=True)
    y_pos = np.arange(len(sorted_data))
    colors = ["red" if x < 0 else "green" for x in sorted_data["improvement"]]
    axes[1, 0].barh(y_pos, sorted_data["improvement"], color=colors, alpha=0.7)
    axes[1, 0].set_yticks(y_pos)
    axes[1, 0].set_yticklabels(
        [f"{row['dataset']}" for _, row in sorted_data.iterrows()]
    )
    axes[1, 0].set_xlabel("ROC-AUC Improvement")
    axes[1, 0].set_title("All HEST Datasets: ROC-AUC Improvement\n(Trimodal - Bimodal)")
    axes[1, 0].axvline(x=0, color="black", linestyle="--", alpha=0.5)
    axes[1, 0].grid(True, alpha=0.3)

    # Plot 4: Distribution of improvements
    if "improvement" in model_comparison.columns:
        axes[1, 1].hist(
            model_comparison["improvement"], bins=10, alpha=0.7, color="blue"
        )
        axes[1, 1].axvline(x=0, color="red", linestyle="--", alpha=0.8)
        axes[1, 1].axvline(
            x=model_comparison["improvement"].mean(),
            color="green",
            linestyle="-",
            alpha=0.8,
            label="Mean",
        )
        axes[1, 1].set_xlabel("ROC-AUC Improvement")
        axes[1, 1].set_ylabel("Number of Datasets")
        axes[1, 1].set_title("Distribution of ROC-AUC Improvements")
        axes[1, 1].legend()
        axes[1, 1].grid(True, alpha=0.3)

    plt.tight_layout()
    plt.show()

    fig.savefig(snakemake.output.plot)
else:
    print("No comparison data available for plotting")
    # Create empty plot
    fig, ax = plt.subplots(1, 1, figsize=(10, 6))
    ax.text(
        0.5,
        0.5,
        "No data available for HEST per-class analysis",
        ha="center",
        va="center",
        transform=ax.transAxes,
        fontsize=16,
    )
    fig.savefig(snakemake.output.plot)

In [ ]:
# Statistical analysis of improvements
if not model_comparison.empty:
    print("=== HEST STATISTICAL SUMMARY ===")

    improvements = model_comparison["improvement"].dropna()

    if len(improvements) > 0:
        print(f"\nROC-AUC Improvements:")
        print(f"Mean improvement: {improvements.mean():.4f}")
        print(f"Median improvement: {improvements.median():.4f}")
        print(f"Std deviation: {improvements.std():.4f}")
        print(
            f"Datasets with positive improvement: {(improvements > 0).sum()}/{len(improvements)}"
        )
        print(f"Max improvement: {improvements.max():.4f}")
        print(f"Min improvement: {improvements.min():.4f}")

        # Class type analysis (Cancer vs Organ)
        for class_type in model_comparison["class_type"].unique():
            class_data = model_comparison[model_comparison["class_type"] == class_type]
            class_improvements = class_data["improvement"].dropna()

            if not class_improvements.empty:
                positive_improvements = (class_improvements > 0).sum()

                print(f"\n{class_type.upper()} Datasets:")
                print(f"  Mean ROC-AUC improvement: {class_improvements.mean():.4f}")
                print(
                    f"  Datasets improved: {positive_improvements}/{len(class_improvements)}"
                )

                # Top improving datasets in this class
                top_improved = class_data.nlargest(3, "improvement")
                print(f"  Top improving datasets:")
                for _, row in top_improved.iterrows():
                    if pd.notna(row["improvement"]):
                        print(f'    {row["dataset"]}: +{row["improvement"]:.4f}')

    # Save detailed results
    model_comparison.to_csv(snakemake.output.analysis, index=False)
    print(f"\nDetailed results saved to: {snakemake.output.analysis}")
else:
    # Create empty output file
    pd.DataFrame().to_csv(snakemake.output.analysis, index=False)
    print("No analysis data available - created empty output file")

# Save CLIP scores if available
if not clip_scores_df.empty:
    clip_scores_df.to_csv(snakemake.output.clip_scores, index=False)
    print(f"CLIP scores saved to: {snakemake.output.clip_scores}")
else:
    pd.DataFrame().to_csv(snakemake.output.clip_scores, index=False)
    print("No CLIP scores available - created empty output file")